# Marketing Campaign - Customer Management Service

In [1]:
import warnings
warnings.filterwarnings('ignore')

#Install Packages
#!python3 -m pip install fastapi --break-system-packages
#!python3 -m pip install nest_asyncio --break-system-packages
#!pip install --upgrade "uvicorn[standard]" --break-system-packages

In [2]:
#Import packages
import nest_asyncio
import uvicorn
import asyncio
import tempfile
import numpy as np
import pandas as pd
import joblib

from fastapi import FastAPI
from uvicorn import Config, Server
from pydantic import BaseModel
from hdfs import InsecureClient
from tensorflow.keras.models import load_model as keras_load_model
from pyspark.sql import SparkSession

In [3]:
# Classes
class Customers(BaseModel):
    customer_id: int
    first_name: str
    last_name: str
    contact_number: str
    telephone_number: str
    email_address: str
    age: int
    default: str
    balance: float
    housing: str
    loan: str
    duration: int
    campaign: int
    previous: int
    job_blue_collar: bool
    job_entrepreneur: bool
    job_housemaid: bool
    job_management: bool
    job_retired: bool
    job_self_employed: bool
    job_services: bool
    job_student: bool
    job_technician: bool
    job_unemployed: bool
    marital_married: bool
    marital_single: bool
    education_secondary: bool
    education_tertiary: bool
    contact_telephone: bool


In [14]:
# Function to pull model from HDFS, store locally and load using keras.
def load_model_from_hdfs(filename):
    try:
        # Connect to HDFS using insecureclient
        client = InsecureClient('http://localhost:9870', user='hduser')

        hdfs_path = f'/marketing_campaign/models/{filename}'

        # Create a temporary local file
        with tempfile.NamedTemporaryFile(suffix='.h5', delete=False) as temp_file:
            # Download model from HDFS
            client.download(hdfs_path, temp_file.name, overwrite=True)

        # Load using keras (avoid recursion)
        model = keras_load_model(temp_file.name)

        # Rebuild metrics
        model.compile(
            optimizer='adam',
            loss='binary_crossentropy',
            metrics=['accuracy']
        )

        print("Model successful loaded from HDFS!")
        model.summary()
        return model
    except Exception as e:
        print("Unable to load model from HDFS")
        print("Error details:", e)

        
# Function to pull model from HDFS, store locally and load using keras.
def load_scaler_from_hdfs(filename):
    try:
        # Connect to HDFS using insecureclient
        client = InsecureClient('http://localhost:9870', user='hduser')

        hdfs_path = f'/marketing_campaign/models/{filename}'

        # Create a temporary local file
        with tempfile.NamedTemporaryFile(suffix='.pkl', delete=False) as temp_file:
            # Download model from HDFS
            client.download(hdfs_path, temp_file.name, overwrite=True)
        # Load scaler using joblib
        scaler = joblib.load(temp_file.name)

        print("Scaler successful loaded from HDFS!")
        return scaler
    except Exception as e:
        print("Unable to load scaler from HDFS")
        print("Error details:", e)
        
        
# Usage
model = load_model_from_hdfs('ann_model.h5')
scaler2 = load_scaler_from_hdfs('scaler.pkl')

def format_data(customer_data):
    try:
        customer_data_df = pd.DataFrame([customer_data])
        customer_data_df.drop(columns=['customer_id','first_name','last_name', 'contact_number', 'telephone_number', 'email_address'], inplace=True)
        customer_data_df.rename(
                columns={
                    'job_blue_collar': 'job_blue-collar',
                    'job_self_employed': 'job_self-employed'
                },
                inplace=True)
        binary_cols = ['default', 'housing', 'loan']
        for col in binary_cols:
            customer_data_df[col] = customer_data_df[col].map({'yes': 1, 'no': 0})

        # All columns with data type "object" or "category"
        categorical_cols = customer_data_df.select_dtypes(include=["object", "category"]).columns.tolist()
        # One-Hot Encode 
        customer_data_df = pd.get_dummies(customer_data_df, columns=categorical_cols, drop_first=True)
        customer_data_scaled = scaler2.transform(customer_data_df)
        
        # Return scaled values
        return customer_data_scaled
    
    except Exception as e:
        print('Error Details: ', e)
        return(e)

    
def log_data_hdfs(results):

    df = spark.createDataFrame([results])

    # Save to specific folder 
    df.write.mode("append") \
        .option("path", "hdfs:///marketing_campaign/sql_data/") \
        .saveAsTable("customers")


Model successful loaded from HDFS!


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         1,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,649 (14.25 KB)

 Trainable params: 3,649 (14.25 KB)

 Non-trainable params: 0 (0.00 B)

Scaler successful loaded from HDFS!


In [15]:
nest_asyncio.apply()

app = FastAPI()

@app.get("/healthcheck")
async def healthcheck():
    return {"status": "OK", "status_message": "Server is up and running!"}

@app.post("/customers")
async def add_customers(customer_details: Customers):
    try:
        # Convert response to dict
        customer_dict = customer_details.dict()
        
        # preprocess data
        scaled_data = format_data(customer_dict)  # should return numpy array
        
        # Make prediction
        prediction = model.predict(scaled_data)
        
        prediction_value = np.round(prediction[0])
    
        result= {
            "status":"Success",
            "customer_id": customer_dict['customer_id'],
            "first_name": customer_dict['first_name'],
            "last_name": customer_dict['last_name'],
            "contact_number": customer_dict['contact_number'],
            "telephone_number": customer_dict['telephone_number'],
            "email_address":customer_dict['email_address'],
            "prediction": int(prediction_value[0])
        }
        log_data_hdfs(result)
        
        return result
    
    except Exception as e:
        return {"error": str(e)}


In [20]:
config = Config(app=app, host="0.0.0.0", port=8000, lifespan="on")
server = Server(config)

await server.serve()


INFO:     Started server process [4394]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step


INFO:     127.0.0.1:44250 - "POST /customers HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [4394]


In [21]:
# Sample Curl Request

# curl -X POST http://localhost:8000/customers \
#   -H "Content-Type: application/json" \
#   -d '{
#     "customer_id": 0,
#     "first_name": "Joel",
#     "last_name": "Montuya",
#     "age": 18,
#     "marital_status": "single",
#     "default": 'no',
#     "housing": 'yes',
#     "loan": 'yes',
#     "duration": 0,
#     "campaign": 0,
#     "previous": 0,
#     "job_blue_collar": false,
#     "job_entrepreneur": false,
#     "job_housemaid": false,
#     "job_management": false,
#     "job_retired": false,
#     "job_self_employed": false,
#     "job_services": false,
#     "job_student": false,
#     "job_technician": false,
#     "job_unemployed": false,
#     "marital_married": false,
#     "marital_single": false,
#     "education_secondary": false,
#     "education_tertiary": false,
#     "contact_telephone": false
#   }'



In [22]:
spark.sql("SHOW TABLES").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|  default|customers|      false|
+---------+---------+-----------+



In [23]:
spark.sql("SELECT customer_id, last_name, first_name, email_address, contact_number, telephone_number, prediction  FROM customers").show()

+-----------+---------+----------+--------------------+--------------+----------------+----------+
|customer_id|last_name|first_name|       email_address|contact_number|telephone_number|prediction|
+-----------+---------+----------+--------------------+--------------+----------------+----------+
|      10121|  Montuya|      Joel|montuya_joel@gmai...|    0838119159|                |         1|
|      12231|    Bribe|     James|  j_brribe@gmail.com|   08381193449|                |         0|
+-----------+---------+----------+--------------------+--------------+----------------+----------+

